In [1]:
!git clone https://github.com/AGupta-23/RetinaScan

Cloning into 'RetinaScan'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 39 (delta 8), reused 37 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 4.37 MiB | 46.12 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [2]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 206MB/s]


In [3]:
for param in model.parameters():
    param.requires_grad = False

In [4]:
for param in model.layer4.parameters():
    param.requires_grad = True

In [5]:
num_features = model.fc.in_features  # 2048 for ResNet50
model.fc = nn.Linear(num_features, 1)  # 1 output unit for binary classification

In [6]:
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

Trainable params: 14,966,785 / 23,510,081 (63.7%)


In [7]:
for name, child in model.named_children():
    trainable = any(p.requires_grad for p in child.parameters())
    print(f"{name:12s} trainable={trainable}")

conv1        trainable=False
bn1          trainable=False
relu         trainable=False
maxpool      trainable=False
layer1       trainable=False
layer2       trainable=False
layer3       trainable=False
layer4       trainable=True
avgpool      trainable=False
fc           trainable=True
